In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
%%capture

# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py
os.chdir(WORKING_DIR)

In [2]:
!pip install optuna
import optuna

In [3]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


In [4]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [5]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

In [8]:
# Define objective function for hyperparameter tuning
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

STUDY_NAME = SLIMElasticNetRecommender.RECOMMENDER_NAME
# l1_ratio=0.1, alpha = 1.0, positive_only=True, topK = 100
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = SLIMElasticNetRecommender(URM_train)
    recommender_instance.fit(
        l1_ratio=optuna_trial.suggest_float("l1_ratio", 0.0, 1.0),
        alpha=optuna_trial.suggest_float("alpha", 1e-6, 1e2, log=True),
        positive_only=optuna_trial.suggest_categorical("positive_only", [True, False]),
        topK=optuna_trial.suggest_int("topK", 10, 1000, step=10)
    )

    return evaluate_recommender(recommender_instance, at=20)

In [10]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=0
)

[I 2025-11-05 21:50:30,536] Using an existing study with name 'SLIMElasticNetRecommender' instead of creating a new one.



Study statistics: 
  Number of finished trials:  52
  Number of pruned trials:  0
  Number of complete trials:  47

Best Value: 0.28934742868996155
Best Params: {'l1_ratio': 0.5342964541170618, 'alpha': 0.0005497715703344824, 'positive_only': False, 'topK': 640}


In [13]:
optuna.visualization.plot_optimization_history(optuna_study)

In [12]:
optuna.visualization.plot_param_importances(optuna_study)

In [14]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [16]:
bp = optuna_study.best_trial.params

def refined_objective(trial):    
    recommender_instance = SLIMElasticNetRecommender(URM_train)
    recommender_instance.fit(
        l1_ratio=trial.suggest_float(
            "l1_ratio",
            max(0.0, bp["l1_ratio"] - 0.2),
            min(1.0, bp["l1_ratio"] + 0.2)
        ),
        alpha=trial.suggest_float(
            "alpha",
            bp["alpha"] / 3,
            bp["alpha"] * 3,
            log=True
        ),
        positive_only=bp["positive_only"],
        topK=trial.suggest_int(
            "topK",
            max(10, bp["topK"] - 50),
            min(1000, bp["topK"] + 50)
        )
    )

    return evaluate_recommender(recommender_instance, at=20)

In [18]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    refined_objective,
    study_name=STUDY_NAME+"_refined",
    n_trials=0
)

[I 2025-11-05 22:34:03,740] Using an existing study with name 'SLIMElasticNetRecommender_refined' instead of creating a new one.



Study statistics: 
  Number of finished trials:  3
  Number of pruned trials:  0
  Number of complete trials:  2

Best Value: 0.2900477994572642
Best Params: {'l1_ratio': 0.6257826858334088, 'alpha': 0.000818987106299432, 'topK': 646}


In [20]:
# Train final model on train + validation with best hyperparameters
recommender = SLIMElasticNetRecommender(URM_train + URM_validation)
recommender.fit(
    l1_ratio=optuna_study.best_trial.params["l1_ratio"],
    alpha=optuna_study.best_trial.params["alpha"],
    positive_only=False,
    topK=optuna_study.best_trial.params["topK"]
)

# Save the trained model
recommender.save_model(paths.MODEL_DIR)

SLIMElasticNetRecommender: Processed 2360 (33.9%) in 5.00 min. Items per second: 7.86
SLIMElasticNetRecommender: Processed 4785 (68.7%) in 10.00 min. Items per second: 7.97
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 14.49 min. Items per second: 8.02
SLIMElasticNetRecommender: Saving model in file '/kaggle/working/modelsSLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete


In [21]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")